# 🚀 All-in-One Gradio TTS App MVP — Qwen3-TTS

A comprehensive 5-tab Gradio web application combining all Qwen3-TTS features into one shareable interactive URL.

Tabs:
1. **Custom Voice** — generate with preset speakers
2. **Voice Design Lab** — design voices with descriptions
3. **Voice Clone** — clone voices from references
4. **Script Builder** — multi-speaker dialogues
5. **Audio Library** — download history


In [ ]:
!pip install -q qwen-tts soundfile gradio
import gradio as gr
import soundfile as sf
import numpy as np
import torch, os, shutil, glob
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "output_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ACTIVE_MODE = "CUSTOM"  # CUSTOM, DESIGN, CLONE

print(f"Loading model for mode: {ACTIVE_MODE}")
model = None
if ACTIVE_MODE == "CUSTOM":
    model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")
elif ACTIVE_MODE == "DESIGN":
    model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")
elif ACTIVE_MODE == "CLONE":
    model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")

def save_audio(wav, sr, prefix="output"):
    idx = len(glob.glob(f"{OUTPUT_DIR}/{prefix}_*.wav"))
    path = f"{OUTPUT_DIR}/{prefix}_{idx}.wav"
    sf.write(path, wav, sr)
    return path

In [ ]:
with gr.Blocks(title="Qwen3-TTS Gradio App") as demo:
    gr.Markdown("## 🚀 Qwen3-TTS All-in-One Platform")
    
    with gr.Tabs():
        with gr.Tab("1. Custom Voice"):
            gr.Markdown("Use preset voices.")
            cv_text = gr.Textbox(lines=5, label="Text")
            cv_speaker = gr.Dropdown(choices=model.get_supported_speakers() if ACTIVE_MODE == "CUSTOM" else ["Ryan"], label="Speaker", value="Ryan")
            cv_lang = gr.Dropdown(choices=["English", "Chinese", "French", "Spanish", "German", "Japanese", "Korean", "Italian", "Portuguese", "Russian"], label="Language", value="English")
            cv_inst = gr.Textbox(label="Instruction (Optional)")
            cv_btn = gr.Button("Generate")
            cv_out = gr.Audio(label="Output")
            
            def gen_cv(text, speaker, lang, inst):
                if ACTIVE_MODE != "CUSTOM": return None
                wav, sr = model.generate_custom_voice(text, lang, speaker, inst)
                return save_audio(wav[0], sr, "custom")
            cv_btn.click(gen_cv, inputs=[cv_text, cv_speaker, cv_lang, cv_inst], outputs=cv_out)

        with gr.Tab("2. Voice Design"):
            gr.Markdown("Design a voice with text prompt.")
            vd_text = gr.Textbox(lines=3, label="Text")
            vd_inst = gr.Textbox(lines=3, label="Voice Description", placeholder="A young confident female voice...")
            vd_lang = gr.Dropdown(choices=["English", "Chinese"], label="Language", value="English")
            vd_btn = gr.Button("Generate")
            vd_out = gr.Audio(label="Output")
            vd_compare = gr.Button("A/B Compare (3 variants)")
            
            def gen_vd(text, inst, lang):
                if ACTIVE_MODE != "DESIGN": return None
                wav, sr = model.generate_voice_design(text, lang, inst)
                return save_audio(wav[0], sr, "design")
            vd_btn.click(gen_vd, inputs=[vd_text, vd_inst, vd_lang], outputs=vd_out)

        with gr.Tab("3. Voice Clone"):
            gr.Markdown("Clone voice from reference.")
            vc_text = gr.Textbox(lines=3, label="Text to Speak")
            vc_ref = gr.Audio(type="filepath", label="Reference Audio")
            vc_ref_text = gr.Textbox(label="Reference Transcript")
            vc_lang = gr.Dropdown(choices=["English", "Chinese"], label="Language", value="English")
            vc_btn = gr.Button("Clone & Generate")
            vc_out = gr.Audio(label="Output")
            
            def gen_vc(text, ref_path, ref_text, lang):
                if ACTIVE_MODE != "CLONE" or not ref_path: return None
                ref_wav, _ = sf.read(ref_path)
                wav, sr = model.generate_voice_clone(text, lang, ref_wav, ref_text)
                return save_audio(wav[0], sr, "clone")
            vc_btn.click(gen_vc, inputs=[vc_text, vc_ref, vc_ref_text, vc_lang], outputs=vc_out)

        with gr.Tab("4. Script Builder"):
            gr.Markdown("WIP: Dialogue generation using CUSTOM mode.")
        
        with gr.Tab("5. Audio Library"):
            gr.Markdown("View and download generated audio.")
            al_gallery = gr.File(label="Generated Files", file_count="multiple")
            al_refresh = gr.Button("Refresh")
            def refresh_lib(): return glob.glob(f"{OUTPUT_DIR}/*.wav")
            al_refresh.click(refresh_lib, outputs=al_gallery)


In [ ]:
demo.launch(share=True, debug=False)

## Mode Switching
To switch modes (e.g. from CUSTOM to DESIGN), restart the runtime or clear VRAM:
```python
import gc
del model
gc.collect()
torch.cuda.empty_cache()
```